This notebook will serve to generate bathymetric surfaces using a TIN method for synthetic SBES, then compare the errors back to MBES.

Error analysis should be done using:
1. Different SBES inter-track sounding spacing
2. Different SBES trackline spacing
3. In the context of bed complexity (e.g., Morphological Variation Index and/or Terrain Ruggedness Index by Wilson)

In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

from interpolators.TIN import TIN

import processing_help
import numpy as np

import xdem
import libpysal as ps
import esda

np.random.seed(1337)

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

In [ ]:
WORK_DIR = Path('/mnt/teamgroup/AutoDBM/Public')
CRS = pd.read_csv(WORK_DIR / 'survey_CRS_list.csv')

In [ ]:
# Build a boundary catalog from SurveyJob layers, then match each survey to the best boundary by SurveyId
data_pairs = {}

def _norm(s):
    return "".join(ch for ch in str(s).upper() if ch.isalnum())

def _tokens(s):
    return {t for t in str(s).upper().replace("-", "_").split("_") if t}

# 1) Read all boundary candidates once
boundary_catalog = []
for gdb_file in WORK_DIR.glob("*/*.gdb"):
    try:
        sj = gpd.read_file(gdb_file, layer="SurveyJob")
    except Exception:
        continue

    if "SurveyId" not in sj.columns or sj.empty:
        continue

    # keep one record per SurveyId (first occurrence)
    for sid, grp in sj.groupby("SurveyId", dropna=True):
        boundary_catalog.append({
            "survey_id": str(sid),
            "survey_norm": _norm(sid),
            "survey_tokens": _tokens(sid),
            "gdb_file": gdb_file,
            "boundary": grp.copy()
        })

# 2) Match each survey from CRS table to best boundary
for _, items in pd.read_csv(crs_data).iterrows():
    survey = items["Survey"]

    sbes_match = [p for p in sbes if survey in str(p)]
    mbes_match = [p for p in mbes if survey in str(p)]
    if not sbes_match or not mbes_match:
        print(f"Missing SBES/MBES file for: {survey}")
        continue

    s_norm = _norm(survey)
    s_tokens = _tokens(survey)

    # exact normalized SurveyId match first
    exact = [c for c in boundary_catalog if c["survey_norm"] == s_norm]

    if exact:
        best = exact[0]
    else:
        # fallback: token overlap score
        scored = []
        for c in boundary_catalog:
            inter = len(s_tokens & c["survey_tokens"])
            denom = max(1, len(s_tokens))
            score = inter / denom
            scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1] if scored and scored[0][0] > 0 else None

    if best is None:
        print(f"No boundary found for: {survey}")
        continue

    boundary = best["boundary"]
    data_pairs[survey] = [
        sbes_match[0],
        mbes_match[0],
        boundary,
    ]

    print(f"{survey} -> {best['survey_id']} ({best['gdb_file'].name})")

In [ ]:
survey_mbes = {}
survey_sbes = {}

for survey in list(data_pairs.keys()):
    survey_mbes[survey] = list(WORK_DIR.glob(f'{survey}/*MBES*.tif'))
    survey_sbes[survey] = list(WORK_DIR.glob(f'{survey}/*SBES*.tif'))

In [ ]:
crs_data = WORK_DIR / 'survey_CRS_list.csv'

geojson = Path(os.getcwd()).parent / 'National_Channel_Framework_-6742069550782745925.geojson'
ncf = gpd.read_file(geojson)

# First Error Metric Analysis: Sounding Spacing
- Along-track individual sounding spacing
- Trackline spacing

In [ ]:
for survey in list(data_pairs.keys()):
    mbes = survey_mbes[survey][0]
    sbes = survey_sbes[survey]

In [ ]:
test_survey = list(data_pairs.keys())[24]
test_mbes = survey_mbes[test_survey][0]
test_sbes = survey_sbes[test_survey]

In [ ]:
mbes_dbm = xdem.DEM(test_mbes)
mbes_dbm.plot()

In [ ]:
sbes_objs = {}
for tif in test_sbes:
    sbes_objs[tif.stem] = xdem.DEM(tif).reproject(mbes_dbm)

In [ ]:
ddbm_objs = {}

for k, v, in sbes_objs.items():
    ddbm = mbes_dbm - v
    # ddbm_stats = ddbm.get_stats()

    # ddbm_objs[k] = (ddbm, ddbm_stats['RMSE'], ddbm_stats['NMAD'], ddbm_stats['90th percentile'])
    ddbm_objs[k] = (ddbm, ddbm.get_stats('rmse'), ddbm.get_stats('nmad'), np.percentile(np.abs(ddbm), 95))

In [ ]:
for k, v in ddbm_objs.items():
    print(k)
    
    print(f"RMSE:{v[1]}\nNMAD:{v[2]}\n95th Percentile |Error|:{v[3]}\n")
v[0].plot(
    cmap="PuOr",
    vmin=float(v[0].get_stats('min')),
    vmax=float(v[0].get_stats('max')),
    cbar_title="Elevation differences (ft)",)

In [ ]:
# TODO: Reproduce the aggregate heat map with my data. Use NMAD instead of MAE, it is better statistic for surface error

# Second Error Metric Analysis: Bed Complexity
- Terrain Ruggedness Index from Wilson, 2007 | $tri_{\textrm{Wilson}} = \textrm{mean}_{ij} (|h_{00} - h_{ij}|) .$

In [ ]:
def get_error_tri(survey_entry: list | None = None):
    import matplotlib.pyplot as plt

    if survey_entry is None:
        raise ValueError("survey_entry is required")

    mbes_tin_path = survey_entry[0]
    sbes_tin_path = survey_entry[1]

    mbes_dbm = xdem.DEM(mbes_tin_path)

    sbes_dbm = xdem.DEM(sbes_tin_path)
    sbes_dbm = sbes_dbm.reproject(mbes_dbm)

    ddbm = mbes_dbm - sbes_dbm
    ddbm_stats = ddbm.get_stats()

    ddbm_abs = np.abs(ddbm)
    ddbm_abs_stats = ddbm_abs.get_stats()

    tri = mbes_dbm.terrain_ruggedness_index(method="Wilson", window_size=3)

    fig, ax = plt.subplots(1,3, figsize=(15,30), dpi = 300)
    tri.plot(ax=ax[0], cmap="Purples", cbar_title="Terrain ruggedness index (ft)")

    ddbm.plot(
        ax=ax[1],
        cmap="PuOr",
        vmin=float(ddbm_stats["Min"]),
        vmax=float(ddbm_stats["Max"]),
        cbar_title="Elevation differences (ft)",
    )

    ddbm_abs.plot(
        ax=ax[2],
        cmap="PuOr",
        vmin=float(ddbm_abs_stats["Min"]),
        vmax=float(ddbm_abs_stats["Max"]),
        cbar_title="Absolute Elevation differences (ft)",
    )

    return mbes_dbm, sbes_dbm, ddbm, ddbm_abs, tri, fig

In [ ]:
survey = list(data_pairs.keys())[15]

mbes_dbm, sbes_dbm, ddbm, ddbm_abs, tri, _ = get_error_tri(data_pairs[survey])

### Some Ideas:
1. OLS or GWR to relate the error with the TRI to show how error is SBES relates with TRI
3. Spatial GLM to do the same as 1, but also including the spacing in soundings
2. Getis-Ord G statistic for hot and cold spots in errors, relate this back to TRI


In [ ]:
from rasterio.transform import from_origin

data = tri.data
transform = tri.transform
width, height = data.shape[1], data.shape[0]

# Initialize lists to store coordinates
x_coords = []
y_coords = []

# Loop through each pixel in the raster
for row in range(height):
    for col in range(width):
        # Calculate x and y using the transform
        x, y = transform * (col + 0.5, row + 0.5)  # Adding 0.5 to get the center of the pixel
        x_coords.append(x)
        y_coords.append(y)

x_coords_array = np.array(x_coords).reshape(height, width)
y_coords_array = np.array(y_coords).reshape(height, width)

In [ ]:
y_arr = ddbm.data.data
x_arr = tri.data.data
mask = ddbm.data.mask & tri.data.mask

# Coordinates of valid cells (projected map units)
xs, ys = np.meshgrid(x_coords_array, y_coords_array)
coords = np.column_stack([xs[mask], ys[mask]])

y = np.abs(y_arr[mask]).reshape(-1, 1)   # |residual| as the response
X = x_arr[mask].reshape(-1, 1)           # TRI as predictor

# 3. Standardize X (recommended for GWR stability)
X = (X - X.mean()) / X.std()

In [ ]:
sbes_gdf

In [ ]:
plt.hist(sbes_gdf.Length_along_transect_ft)

In [ ]:
# TODO: Make a function to sample based on "Length_along_transect_ft"

# Dredge Cut Estimation

**NOTE:** This may not be included in the TN

In [ ]:
methods = {"MBES": mbes_tin_path ,'SBES': sbes_tin_path}
tgt_ncf = ncf[ncf['channelareaidfk'].str.contains(f'{survey[:9]}')]
tgt_ncf

In [ ]:
unit = "yd"

reach_vols = {}

for id, geom in tgt_ncf.iterrows():
    reachid = geom.channelreachidpk
    reach_auth_depth = geom.depthauthorized
    # print(reach_auth_depth)
    reach_vols[reachid] = {name: processing_help.compute_volume(path, tgt_ncf, auth_depth = reach_auth_depth, unit = unit) for name, path in methods.items()}

for reach, volumes in reach_vols.items():
    print(f'\n{reach}')
    print("_"*50)
    for key, value in volumes.items():
        print(f'{key}: {value:,.1f} {unit}³')

In [ ]:
# Differences
for reach, volumes in reach_vols.items():
    print(f'\n{reach}')
    print(f"_"*50)
    for a, b in [('MBES', 'SBES')]:
        diff = volumes[a] - volumes[b]
        print(f'{a} - {b}: {diff:,.1f} {unit}³')